# Calculate New Parameter Sweep Parameters

This notebook calculates the necessary parameters for the modified parameter sweep:

1. **Aspect Ratios**: 4 different a/b ratios (6.88, ~7.95, ~9.03, 10.1)
2. **Muscle Dimensions**: Calculate b and a for each (Volume, aspect_ratio) combination
3. **Force Adjustment**: Calculate actual force values to apply 0-45N relative to surface area
4. **Am/Rho Sampling**: Generate 4 random combinations from available values

## Background

The issue: With `divideNeumannBoundaryConditionValuesByTotalArea = True`, larger surface areas received less stress (force per area) for the same nominal force. To ensure all geometries reach comparable stress levels up to 45N (referenced to surface area), we need to adjust the actual force values based on each geometry's surface area.

In [5]:
import pandas as pd
import numpy as np
import random
from pathlib import Path

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

## 1. Define Aspect Ratios

Cuboid geometry: [b, a, a] where b = muscle length (y-direction), a = cross-section
Aspect ratio: **b/a** (length / cross-section)

Target ratios: 6.88 (min), 10.1 (max), plus 2 equally-spaced middle values

**Note**: This matches the old script where muscle_y (length) / muscle_x (cross-section) ranged from ~9.1

In [11]:
# Define aspect ratios (b/a) - length / cross-section
min_ratio = 6.88
max_ratio = 10.1
n_ratios = 4

# Calculate equally spaced ratios
aspect_ratios = np.linspace(min_ratio, max_ratio, n_ratios)

print("Aspect Ratios (b/a - length/cross-section):")
for i, ratio in enumerate(aspect_ratios):
    print(f"  {i+1}. {ratio:.4f}")

# Store for later
aspect_ratios_list = aspect_ratios.tolist()

Aspect Ratios (b/a - length/cross-section):
  1. 6.8800
  2. 7.9533
  3. 9.0267
  4. 10.1000


## 2. Calculate Muscle Dimensions for Each (Volume, Aspect Ratio) Combination

For a cuboid [b, a, a]:
- Volume = b × a²
- Aspect ratio r = b/a (length / cross-section), so b = r × a
- Volume = (r × a) × a² = a³ × r
- Therefore: a = (Volume / r)^(1/3)
- And: b = r × a

Also: Surface area A_top = a² = Volume / b

In [7]:
# Define volumes (reduced set)
volumes = [421.6, 455.9, 573.5, 691.2, 738.0]

print(f"\nVolumes: {volumes}")
print(f"Number of volumes: {len(volumes)}")
print(f"Number of aspect ratios: {len(aspect_ratios)}")
print(f"Total (Volume, Ratio) combinations: {len(volumes) * len(aspect_ratios)}")


Volumes: [421.6, 455.9, 573.5, 691.2, 738.0]
Number of volumes: 5
Number of aspect ratios: 4
Total (Volume, Ratio) combinations: 20


In [9]:
# Calculate dimensions for each combination
dimension_data = []

for volume in volumes:
    for ratio in aspect_ratios:
        # Calculate a and b
        # ratio = b/a, so b = ratio * a
        # Volume = b * a² = (ratio * a) * a² = ratio * a³
        # Therefore: a = (Volume / ratio)^(1/3)
        a = (volume / ratio) ** (1/3)
        b = ratio * a
        
        # Verify volume
        calculated_volume = b * a * a
        
        # Calculate surface area
        surface_area = a * a
        
        dimension_data.append({
            'volume_cm3': volume,
            'aspect_ratio_b_a': ratio,
            'muscle_extent_y_b_cm': b,
            'muscle_extent_x_a_cm': a,
            'top_surface_area_cm2': surface_area,
            'volume_check_cm3': calculated_volume
        })

df_dimensions = pd.DataFrame(dimension_data)

print("\nMuscle Dimensions:")
print(df_dimensions.to_string(index=False))


# Verify volumes matchprint(f"\nMaximum volume calculation error: {max_vol_error:.10f} cm³")
max_vol_error = (df_dimensions['volume_cm3'] - df_dimensions['volume_check_cm3']).abs().max()


Muscle Dimensions:
 volume_cm3  aspect_ratio_b_a  muscle_extent_y_b_cm  muscle_extent_x_a_cm  top_surface_area_cm2  volume_check_cm3
      421.6          6.880000             27.124339              3.942491             15.543236             421.6
      421.6          7.953333             29.876725              3.756504             14.111319             421.6
      421.6          9.026667             32.507617              3.601287             12.969268             421.6
      421.6         10.100000             35.035995              3.468910             12.033339             421.6
      455.9          6.880000             27.840829              4.046632             16.375231             455.9
      455.9          7.953333             30.665919              3.855732             14.866667             455.9
      455.9          9.026667             33.366306              3.696415             13.663484             455.9
      455.9         10.100000             35.961471              3.5

## 3. Calculate Force Values Adjusted for Surface Area

**Problem**: With `divideNeumannBoundaryConditionValuesByTotalArea = True`, the same nominal force produces different stress on different surface areas.

**Goal**: Apply forces equivalent to 0-45N in 3N steps, **referenced to a baseline surface area**.

**Strategy**:
1. Choose a reference surface area (use minimum area as baseline)
2. For each geometry, calculate what actual force is needed to produce the same stress as [0, 3, 6, ..., 45]N at the reference area
3. Formula: F_actual = (F_target / A_reference) × A_geometry
   - This ensures: F_actual / A_geometry = F_target / A_reference (same stress)

In [5]:
# Define target force range (referenced to baseline surface area)
force_target_values = list(range(0, 46, 3))  # 0, 3, 6, ..., 45 N

print(f"Target force values (at reference area): {force_target_values}")
print(f"Number of force values: {len(force_target_values)}")

Target force values (at reference area): [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36, 39, 42, 45]
Number of force values: 16


In [10]:
# Choose reference surface area (minimum area = baseline with highest stress)
reference_area_cm2 = df_dimensions['top_surface_area_cm2'].min()

print(f"\nReference surface area (minimum): {reference_area_cm2:.4f} cm²")
print(f"Maximum surface area: {df_dimensions['top_surface_area_cm2'].max():.4f} cm²")
print(f"Surface area range: {df_dimensions['top_surface_area_cm2'].min():.4f} - {df_dimensions['top_surface_area_cm2'].max():.4f} cm²")


Reference surface area (minimum): 12.0333 cm²
Maximum surface area: 22.5759 cm²
Surface area range: 12.0333 - 22.5759 cm²


In [3]:
# Calculate actual force values for each geometry
force_data = []

for _, row in df_dimensions.iterrows():
    volume = row['volume_cm3']
    ratio = row['aspect_ratio_b_a']
    b = row['muscle_extent_y_b_cm']
    a = row['muscle_extent_x_a_cm']
    area = row['top_surface_area_cm2']
    
    for f_target in force_target_values:
        # Calculate actual force needed to achieve target stress
        # Stress_target = F_target / A_reference
        # We want: F_actual / A_geometry = F_target / A_reference
        # Therefore: F_actual = F_target × (A_geometry / A_reference)
        
        f_actual = f_target * (area / reference_area_cm2)
        
        # Calculate the stress that will be applied
        stress = f_actual / area  # This should equal f_target / reference_area_cm2
        
        force_data.append({
            'volume_cm3': volume,
            'aspect_ratio_b_a': ratio,
            'muscle_extent_y_b_cm': b,
            'muscle_extent_x_a_cm': a,
            'top_surface_area_cm2': area,
            'force_target_N': f_target,
            'force_actual_N': f_actual,
            'stress_N_cm2': stress
        })

df_forces = pd.DataFrame(force_data)

print("\nForce Calculations (first 20 rows):")
print(df_forces.head(20).to_string(index=False))

NameError: name 'df_dimensions' is not defined

In [11]:
# Verify that stress is consistent across geometries for same target force
print("\nVerification: Stress should be identical for same target force across all geometries")
print("="*80)

for f_target in [0, 15, 30, 45]:
    subset = df_forces[df_forces['force_target_N'] == f_target]
    print(f"\nTarget Force: {f_target} N")
    print(f"  Stress range: {subset['stress_N_cm2'].min():.8f} - {subset['stress_N_cm2'].max():.8f} N/cm²")
    print(f"  Stress std dev: {subset['stress_N_cm2'].std():.10f} N/cm²")
    print(f"  All equal? {subset['stress_N_cm2'].nunique() == 1}")


Verification: Stress should be identical for same target force across all geometries

Target Force: 0 N
  Stress range: 0.00000000 - 0.00000000 N/cm²
  Stress std dev: 0.0000000000 N/cm²
  All equal? True

Target Force: 15 N
  Stress range: 0.07375054 - 0.07375054 N/cm²
  Stress std dev: 0.0000000000 N/cm²
  All equal? False

Target Force: 30 N
  Stress range: 0.14750109 - 0.14750109 N/cm²
  Stress std dev: 0.0000000000 N/cm²
  All equal? False

Target Force: 45 N
  Stress range: 0.22125163 - 0.22125163 N/cm²
  Stress std dev: 0.0000000000 N/cm²
  All equal? False


In [ ]:
# Show force scaling for different geometries at same target
print("\nExample: How actual force varies with geometry for target force = 30N")
print("="*80)

example_forces = df_forces[df_forces['force_target_N'] == 30].copy()
example_forces = example_forces.sort_values(['volume_cm3', 'aspect_ratio_b_a'])

print(example_forces[['volume_cm3', 'aspect_ratio_b_a', 'top_surface_area_cm2', 
                       'force_target_N', 'force_actual_N', 'stress_N_cm2']].to_string(index=False))

print(f"\nActual force range for target 30N: {example_forces['force_actual_N'].min():.2f} - {example_forces['force_actual_N'].max():.2f} N")
print(f"Maximum actual force: {df_forces['force_actual_N'].max():.2f} N (at target 45N, largest area)")

## 4. Generate Random Am/Rho Combinations

Instead of all 12 combinations (3 Am × 4 Rho), randomly sample 4 combinations.

In [12]:
# Available values
am_values = [450, 500, 550]
rho_values = [10.493, 10.534, 10.575, 10.616]

# Generate all possible combinations
all_combinations = [(am, rho) for am in am_values for rho in rho_values]

print(f"All possible (Am, Rho) combinations: {len(all_combinations)}")
print("All combinations:")
for am, rho in all_combinations:
    print(f"  Am={am}, Rho={rho}")

# Randomly sample 4 combinations
random_combinations = random.sample(all_combinations, 4)

print(f"\nRandomly selected 4 combinations:")
for i, (am, rho) in enumerate(random_combinations, 1):
    print(f"  {i}. Am={am}, Rho={rho}")

# Store for export
am_selected = [am for am, rho in random_combinations]
rho_selected = [rho for am, rho in random_combinations]

All possible (Am, Rho) combinations: 12
All combinations:
  Am=450, Rho=10.493
  Am=450, Rho=10.534
  Am=450, Rho=10.575
  Am=450, Rho=10.616
  Am=500, Rho=10.493
  Am=500, Rho=10.534
  Am=500, Rho=10.575
  Am=500, Rho=10.616
  Am=550, Rho=10.493
  Am=550, Rho=10.534
  Am=550, Rho=10.575
  Am=550, Rho=10.616

Randomly selected 4 combinations:
  1. Am=550, Rho=10.575
  2. Am=450, Rho=10.534
  3. Am=450, Rho=10.493
  4. Am=500, Rho=10.493


## 5. Summary: Parameters for New Parameter Sweep

In [8]:
print("="*80)
print("PARAMETER SWEEP CONFIGURATION")
print("="*80)

print(f"\n1. ASPECT RATIOS (a/b): {len(aspect_ratios)} values")
for i, ratio in enumerate(aspect_ratios, 1):
    print(f"   {i}. {ratio:.4f}")

print(f"\n2. VOLUMES [cm³]: {len(volumes)} values")
for vol in volumes:
    print(f"   - {vol}")

print(f"\n3. FORCE TARGET VALUES [N]: {len(force_target_values)} values")
print(f"   Range: {min(force_target_values)} - {max(force_target_values)} N in 3N steps")
print(f"   Values: {force_target_values}")

print(f"\n4. AM/RHO COMBINATIONS: {len(random_combinations)} random samples")
for i, (am, rho) in enumerate(random_combinations, 1):
    print(f"   {i}. Am={am} cm⁻¹, Rho={rho} ×10⁻⁴ kg/cm³")

print(f"\n5. REFERENCE SURFACE AREA: {reference_area_cm2:.4f} cm²")

print("\n" + "="*80)
print("TOTAL SIMULATIONS")
print("="*80)
total_sims = len(aspect_ratios) * len(volumes) * len(force_target_values) * len(random_combinations)
print(f"Aspect Ratios × Volumes × Forces × (Am,Rho) Combinations")
print(f"{len(aspect_ratios)} × {len(volumes)} × {len(force_target_values)} × {len(random_combinations)} = {total_sims} simulations")

# Compare to old configuration
old_total = 12 * 5 * 7 * 12 * 4  # Forces × muscle_y × volumes × (Am×Rho)
print(f"\nOld configuration: {old_total} simulations")
print(f"New configuration: {total_sims} simulations")
print(f"Reduction: {old_total - total_sims} fewer simulations ({100*(1 - total_sims/old_total):.1f}% reduction)")

PARAMETER SWEEP CONFIGURATION

1. ASPECT RATIOS (a/b): 4 values
   1. 6.8800
   2. 7.9533
   3. 9.0267
   4. 10.1000

2. VOLUMES [cm³]: 5 values
   - 421.6
   - 455.9
   - 573.5
   - 691.2
   - 738.0

3. FORCE TARGET VALUES [N]: 16 values
   Range: 0 - 45 N in 3N steps
   Values: [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36, 39, 42, 45]

4. AM/RHO COMBINATIONS: 4 random samples
   1. Am=550 cm⁻¹, Rho=10.575 ×10⁻⁴ kg/cm³
   2. Am=450 cm⁻¹, Rho=10.534 ×10⁻⁴ kg/cm³
   3. Am=450 cm⁻¹, Rho=10.493 ×10⁻⁴ kg/cm³
   4. Am=500 cm⁻¹, Rho=10.493 ×10⁻⁴ kg/cm³

5. REFERENCE SURFACE AREA: 203.3883 cm²

TOTAL SIMULATIONS
Aspect Ratios × Volumes × Forces × (Am,Rho) Combinations
4 × 5 × 16 × 4 = 1280 simulations

Old configuration: 20160 simulations
New configuration: 1280 simulations
Reduction: 18880 fewer simulations (93.7% reduction)


## 6. Export Data for Bash Script

In [2]:
# Save force mapping to CSV for reference
df_forces.to_csv('force_mapping_new_sweep.csv', index=False)
print(f"Saved force mapping to: force_mapping_new_sweep.csv")

# Save dimension calculations
df_dimensions.to_csv('dimension_calculations_new_sweep.csv', index=False)
print(f"Saved dimension calculations to: dimension_calculations_new_sweep.csv")

NameError: name 'df_forces' is not defined

In [ ]:
# Create bash-friendly format for the script
print("\n" + "="*80)
print("BASH SCRIPT VALUES")
print("="*80)

print("\n# Aspect ratios (b/a) - length/cross-section")
print("ASPECT_RATIOS=(" + " ".join([f"{r:.4f}" for r in aspect_ratios]) + ")")

print("\n# Volumes (cm³)")
print("VOLUMES=(" + " ".join([str(v) for v in volumes]) + ")")

print("\n# Force target values (N) - referenced to minimum surface area")
print("FORCE_TARGETS=(" + " ".join([str(f) for f in force_target_values]) + ")")

print("\n# Reference surface area (cm²)")
print(f"REFERENCE_AREA={reference_area_cm2:.6f}")

print("\n# Random Am/Rho combinations")
print("AM_VALUES=(" + " ".join([str(am) for am in am_selected]) + ")")
print("RHO_VALUES=(" + " ".join([str(rho) for rho in rho_selected]) + ")")

print("\n# Total simulations")
print(f"# Total: {total_sims}")


BASH SCRIPT VALUES

# Aspect ratios (a/b)
ASPECT_RATIOS=(6.8800 7.9533 9.0267 10.1000)

# Volumes (cm³)
VOLUMES=(421.6 455.9 573.5 691.2 738.0)

# Force target values (N) - referenced to minimum surface area
FORCE_TARGETS=(0 3 6 9 12 15 18 21 24 27 30 33 36 39 42 45)

# Reference surface area (cm²)
REFERENCE_AREA=203.388328

# Random Am/Rho combinations
AM_VALUES=(550 450 450 500)
RHO_VALUES=(10.575 10.534 10.493 10.493)

# Total simulations
# Total: 1280


## 7. Lookup Functions for Bash Script

The bash script will need to:
1. Calculate b and a from (volume, aspect_ratio)
2. Calculate actual force from (target_force, surface_area)

In [ ]:
# Show the formulas that will be used in bash
print("="*80)
print("FORMULAS FOR BASH SCRIPT")
print("="*80)

print("""
# Given: volume (V), aspect_ratio (r)
# Calculate muscle dimensions:

b = (V / r²)^(1/3)
a = r × b

# Verify: V = b × a²

# Calculate surface area:
A = a²

# Given: force_target (F_target), surface_area (A), reference_area (A_ref)
# Calculate actual force to apply:

F_actual = F_target × (A / A_ref)

# This ensures the stress is consistent:
# stress = F_actual / A = F_target / A_ref
""")

In [ ]:
# Test the formulas with an example
print("\n" + "="*80)
print("EXAMPLE CALCULATION")
print("="*80)

test_volume = 573.5
test_ratio = 7.9533  # b/a
test_force_target = 30

# Calculate dimensions
test_a = (test_volume / test_ratio) ** (1/3)
test_b = test_ratio * test_a
test_area = test_a ** 2

# Calculate actual force
test_force_actual = test_force_target * (test_area / reference_area_cm2)

print(f"\nInput:")
print(f"  Volume: {test_volume} cm³")
print(f"  Aspect ratio (b/a): {test_ratio:.4f}")
print(f"  Target force: {test_force_target} N")

print(f"\nCalculated:")
print(f"  a (muscle_extent_x): {test_a:.6f} cm")
print(f"  b (muscle_extent_y): {test_b:.6f} cm")
print(f"  Surface area: {test_area:.6f} cm²")
print(f"  Actual force to apply: {test_force_actual:.6f} N")

print(f"\nVerification:")
print(f"  Volume check: {test_b * test_a * test_a:.6f} cm³ (should be {test_volume})")
print(f"  Ratio check: {test_b / test_a:.6f} (should be {test_ratio})")
print(f"  Stress: {test_force_actual / test_area:.8f} N/cm²")
print(f"  Reference stress: {test_force_target / reference_area_cm2:.8f} N/cm²")
print(f"  Stress match: {abs(test_force_actual / test_area - test_force_target / reference_area_cm2) < 1e-10}")


EXAMPLE CALCULATION

Input:
  Volume: 573.5 cm³
  Aspect ratio (a/b): 7.9533
  Target force: 30 N

Calculated:
  b (muscle_extent_y): 2.085193 cm
  a (muscle_extent_x): 16.584164 cm
  Surface area: 275.034510 cm²
  Actual force to apply: 40.567890 N

Verification:
  Volume check: 573.500000 cm³ (should be 573.5)
  Stress: 0.14750109 N/cm²
  Reference stress: 0.14750109 N/cm²
  Stress match: True


In [1]:
# Quick verification: Compare with old script values
print("="*80)
print("COMPARISON WITH OLD SCRIPT")
print("="*80)

# Old script example: Volume=421.6, muscle_y=32.69
old_volume = 421.6
old_muscle_y = 32.69
old_muscle_x = (old_volume / old_muscle_y) ** 0.5
old_ratio = old_muscle_y / old_muscle_x

print(f"\nOLD SCRIPT (Volume={old_volume} cm³):")
print(f"  muscle_y (length): {old_muscle_y:.2f} cm")
print(f"  muscle_x (cross-section): {old_muscle_x:.2f} cm")
print(f"  Ratio (y/x): {old_ratio:.2f}")
print(f"  Surface area: {old_muscle_x**2:.2f} cm²")

# New script with similar ratio
new_ratio = old_ratio  # Should be around 9.1
new_volume = old_volume
new_a = (new_volume / new_ratio) ** (1/3)
new_b = new_ratio * new_a

print(f"\nNEW SCRIPT (Volume={new_volume} cm³, ratio={new_ratio:.2f}):")
print(f"  a (cross-section): {new_a:.2f} cm")  
print(f"  b (length): {new_b:.2f} cm")
print(f"  Ratio (b/a): {new_b/new_a:.2f}")
print(f"  Surface area: {new_a**2:.2f} cm²")

print(f"\nDIFFERENCE:")
print(f"  Length: old={old_muscle_y:.2f} vs new={new_b:.2f} cm (diff={abs(old_muscle_y-new_b):.2f})")
print(f"  Cross-section: old={old_muscle_x:.2f} vs new={new_a:.2f} cm (diff={abs(old_muscle_x-new_a):.2f})")
print(f"  Should be very similar!")

COMPARISON WITH OLD SCRIPT

OLD SCRIPT (Volume=421.6 cm³):
  muscle_y (length): 32.69 cm
  muscle_x (cross-section): 3.59 cm
  Ratio (y/x): 9.10
  Surface area: 12.90 cm²

NEW SCRIPT (Volume=421.6 cm³, ratio=9.10):
  a (cross-section): 3.59 cm
  b (length): 32.69 cm
  Ratio (b/a): 9.10
  Surface area: 12.90 cm²

DIFFERENCE:
  Length: old=32.69 vs new=32.69 cm (diff=0.00)
  Cross-section: old=3.59 vs new=3.59 cm (diff=0.00)
  Should be very similar!
